In [1]:
!pip install terratorch==1.1.1

  Using cached terratorch-1.1.1-py3-none-any.whl.metadata (13 kB)
  Using cached albumentations-1.4.10-py3-none-any.whl.metadata (38 kB)
  Using cached albucore-0.0.16-py3-none-any.whl.metadata (3.1 kB)
  Using cached torchmetrics-1.8.2-py3-none-any.whl.metadata (22 kB)
  Using cached geopandas-1.1.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached lightly-1.5.22-py3-none-any.whl.metadata (38 kB)
  Using cached jsonargparse-4.35.0-py3-none-any.whl.metadata (12 kB)
  Using cached lightning-2.6.1-py3-none-any.whl.metadata (44 kB)
  Using cached segmentation_models_pytorch-0.5.0-py3-none-any.whl.metadata (17 kB)
  Using cached hydra_core-1.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached antlr4-python3-runtime-4.9.3.tar.gz (117 kB)
  Preparing metadata (setup.py) ... done
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached nvidia_c

In [2]:
!pip install gdown tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [gdown]32m2/4 [beautifulsoup4]


In [3]:
# import geopandas as gpd
import numpy as np
import pandas as pd
import random
import rasterio as rio
import re
import torch

from glob import glob
from pathlib import Path
# from rasterio.windows import Window, bounds, from_bounds
from shapely.geometry import Point, Polygon
from shapely.ops import unary_union
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader

import torch.nn as nn
from torchvision import transforms
from terratorch.models import necks

import matplotlib.pyplot as plt
from rasterio.plot import show



import os

/users/PGS0218/julina/.conda/envs/lora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data loading

In [4]:
class BeforeData(Dataset):
    def __init__(self,
                before_dir,
                label_dir, 
                input_size = (224, 224), # what size tifs would you like TerraMind to process? #Channels and batch size are handled separately
                num_grabs = 5 # how many times do you want to randomly grab a square of (input size x input size) from each input tif
                ):
        
        # we should do any resizing here
        # Albumentations
        # Figure out where to do before/after and how to handle them... if those will be different iterations of the train or what...
        # Do class relabels
        # Do mean/std deviation editing here

        self.before_files = sorted(Path(before_dir).glob("*.tif"))
        self.label_files = sorted(Path(label_dir).glob("*.tif"))
        self.input_size = input_size 
        self.num_grabs = num_grabs


        self.random_crop = transforms.RandomCrop(self.input_size)

        assert len(self.before_files) == len(self.label_files), "Mismatch in number of before images and labels"

    def __len__(self):
        return len(self.before_files) * self.num_grabs

    def __getitem__(self, index):
        with rio.open(self.before_files[index]) as src_x, rio.open(self.label_files[index]) as src_y :
            x = torch.from_numpy(src_x.read())#.astype("float32")  # shape: (bands, H, W)
            y = torch.from_numpy(src_y.read())#.astype("int64")

            for _ in range(self.num_grabs):
                i, j, h, w = transforms.RandomCrop.get_params(x, output_size=self.input_size)
                x = F.crop(x, i, j, h, w).float().unsqueeze(0) #get rid of unsqueeze later when I do batches properly
                y = F.crop(y, i, j, h, w).int()

            
                return x, y

    

dataset = BeforeData(os.getcwd()+ "/Images/Before", os.getcwd() + "/Images/Label")

x, y = dataset[0]
print(x.shape, y.shape)

IndexError: list index out of range

In [6]:
x.shape[1:]

torch.Size([12, 224, 224])

In [523]:
rand_crop = transforms.RandomCrop((224, 224))
x_reshaped = rand_crop(torch.tensor(x)).unsqueeze(0)
print(x_reshaped.shape)

print(len(x.shape))




torch.Size([1, 12, 224, 224])
3


/var/folders/v6/pv2sv4yx3_q6j2_n2pr5wfkm0000gn/T/ipykernel_33718/1670710981.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_reshaped = rand_crop(torch.tensor(x)).unsqueeze(0)


In [503]:
for i in range(2):
    print(i)

0
1


# Model Creation

In [5]:
from terratorch.registry import BACKBONE_REGISTRY, TERRATORCH_DECODER_REGISTRY

### ENCODER + Neck

In [8]:
class TerraMindEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = BACKBONE_REGISTRY.build(
            'terramind_v1_base', 
            pretrained = True, 
            modalities = ['S2L2A']
        )

            # Neck pipeline
    
        self.select_indices = necks.SelectIndices(
            channel_list=[768, 768, 768, 768, 768],
            indices=[3, 5, 7, 9, 11]
        )

        self.reshape_tokens = necks.ReshapeTokensToImage(
            channel_list=[768, 768, 768, 768, 768],
            remove_cls_token=False
        )

        # Project the representations to fit as the skip layers which will connect to 
        self.projections = nn.ModuleList([
            nn.Conv2d(768, 64, kernel_size=1), #index 3
            nn.Conv2d(768, 128, kernel_size=1), #index 5
            nn.Conv2d(768, 256, kernel_size=1), #index 7
            nn.Conv2d(768, 512, kernel_size=1), #index 9
            nn.Conv2d(768, 1024, kernel_size=1), #index 11
        ])

        # Per-level upsampling
        self.upsamplers = nn.ModuleList([
            nn.Upsample(scale_factor=16, mode="bilinear", align_corners=False),  #index 3
            nn.Upsample(scale_factor=8, mode="bilinear", align_corners=False),  #index 5
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False),  #index 7
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  #index 9
            nn.Identity(),  #index 11
        ])


    def forward(self, x):
        embeddings = self.model(x)
        features = self.select_indices(embeddings)      # list of 4 tensors
        features = self.reshape_tokens(features)           # reshape each to (B, C, H, W)

        out = []
        for f, proj, up in zip(features, self.projections, self.upsamplers):
            f = proj(f) 
            f = up(f)
            out.append(f)
        return out 


model = TerraMindEncoder()
# print(model)
print()


###HAVE TO RESHAPE THE SIZE HERE USING THE RANDOM CROPS
### PROBABLY WILL WANT TO ADD PADDING
### WILL WANT TO MOVE THIS TO DATA MODULE AND WILL WANT TO ADD OTHER TRANSFORMATIONS HERE
rand_crop = transforms.RandomCrop((224, 224))
x_reshaped = rand_crop(torch.tensor(x))
print(x_reshaped.size())
#run model and get representations
og_output = model({'S2L2A': x_reshaped.to(torch.float32)}) #should we be expecting a different representation per band?
print("len:", len(og_output))
for i in og_output:
    print( "shape:", i.shape)



torch.Size([1, 12, 224, 224])
len: 12


/var/folders/v6/pv2sv4yx3_q6j2_n2pr5wfkm0000gn/T/ipykernel_39556/579774449.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_reshaped = rand_crop(torch.tensor(x))


shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])
shape: torch.Size([1, 196, 768])


### DECODER

In [338]:
# help(TERRATORCH_DECODER_REGISTRY['UNetDecoder'])

In [598]:
class UNet2D(nn.Module):
    def __init__(self, output_channels = 3):
        super().__init__()

        self.relu = nn.ReLU()
        # block 4 up
        # up_conv0 
        # index 11 from (1, 1024 ,14, 14)--> (1, 512, 28,28,)
        self.up_conv0 = nn.ConvTranspose2d(in_channels=1024, out_channels=512, kernel_size=2, stride=2)


        # block 3 up
        # concat1 + conv1a + conv1b + up_conv1
        # concat1 = index 9 + up_conv0 (1,512, 28,28,)+(1,512, 28, 28)--> (1,1024, 28, 28)
        # conv1a =  (1, 1024, 28, 28)--> (1, 512, 28, 28)
        # conv1b = (1, 512, 28, 28)--> (1, 512, 28, 28)
        # up_conv1 =  (1, 512, 28, 28) --> (1, 256, 56, 56)
        self.conv2d_1a = nn.Conv2d(1024, 512, kernel_size=3, padding = 1)
        self.conv2d_1b = nn.Conv2d(512, 512, kernel_size=3, padding = 1)
        self.up_conv1 = nn.ConvTranspose2d(in_channels=512, out_channels=256, kernel_size=2, stride=2)
        

        # block 2 up
        # concat2 + conv2a + conv2b + upconv2 = 
        # concat2 = index 7 + upconv1 (1,256, 56, 56)+(1,256, 56, 56)--> (1,512, 56, 56)
        # conv2a =  (1,512, 56, 56) --> (1, 256, 56, 56)
        # conv2b =  (1,256, 56, 56) --> (1,256, 56, 56)
        # upconv2 = (1, 128, 112, 112)
        self.conv2d_2a = nn.Conv2d(512, 256, kernel_size=3, padding = 1)
        self.conv2d_2b = nn.Conv2d(256, 256, kernel_size=3, padding = 1)
        self.up_conv2 = nn.ConvTranspose2d(in_channels=256, out_channels=128, kernel_size=2, stride=2)



        # block 1 up
        # concat3 + conv3a + conv3b + upconv3 = 
        # concat3 = index5 + upconv2 (1, 128, 112, 112) + (1, 128, 112, 112)--> (1, 256, 112, 112)
        # conv3a = (1, 256, 112, 112) --> (1, 128, 112, 112)
        # conv3b = (1, 128, 112, 112) --> (1, 128, 112, 112)
        # upconv3 = (1, 128, 112, 112) --> (1, 64, 224, 224)
        self.conv2d_3a = nn.Conv2d(256, 128, kernel_size=3, padding =1 )
        self.conv2d_3b = nn.Conv2d(128, 128, kernel_size=3, padding =1)
        self.up_conv3 = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=2, stride=2)


        # block 0 up
        # concat4 + conv4a + conv4b + conv_to_output = 
        # concat4 = index3 + upconv3 (1, 64, 224, 224) + (1, 64, 224, 224) --> (1, 128, 224, 224)
        # conv4a = (1, 128, 224, 224) --> (1, 64, 224, 224)
        # conv4b = (1, 64, 224, 224) --> (1, 64, 224, 224)
        # conv4c = (1, 64, 224, 224) --> (1, 3, 224, 224)
        self.conv2d_4a = nn.Conv2d(128, 64, kernel_size=3, padding =1)
        self.conv2d_4b = nn.Conv2d(64, 64, kernel_size=3, padding =1)
        self.conv2d_4c = nn.Conv2d(64, output_channels, kernel_size=1)

    def forward(self, feature_list):
        index3, index5, index7, index9, index11 = feature_list 

        # block 5
        x = self.up_conv0(index11)

        # block 4
        x = torch.cat([x, index9], 1)
        x = self.relu(self.conv2d_1a(x))
        x = self.relu(self.conv2d_1b(x))
        x = self.up_conv1(x)

        # block 3 
        x = torch.cat([x, index7], 1)
        x = self.relu(self.conv2d_2a(x))
        x = self.relu(self.conv2d_2b(x))
        x = self.up_conv2(x)

        # block 2
        x = torch.cat([x, index5], 1)
        x = self.relu(self.conv2d_3a(x))
        x = self.relu(self.conv2d_3b(x))
        x = self.up_conv3(x)

        # block 1 
        x = torch.cat([x, index3], 1)
        x = self.relu(self.conv2d_4a(x))
        x = self.relu(self.conv2d_4b(x))
        logits = self.conv2d_4c(x)
    



        return logits

decoder = UNet2D()
output = decoder(og_output)

print(output.shape)


torch.Size([1, 3, 224, 224])


# Creating Train Loop

In [464]:
encoder = TerraMindEncoder()
decoder = UNet2D()
decoder(encoder({'S2L2A': x_reshaped.to(torch.float32)})).shape


torch.Size([1, 3, 224, 224])

In [474]:
import torch.optim as optim


In [483]:
# Utilities

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        """
        preds: (N, C, H, W) after sigmoid/softmax
        targets: (N, C, H, W) one-hot encoded or same shape
        """
        # flatten
        preds = preds.contiguous().view(preds.shape[0], -1)
        targets = targets.contiguous().view(targets.shape[0], -1)

        intersection = (preds * targets).sum(dim=1)
        dice_score = (2. * intersection + self.smooth) / (
            preds.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )

        return 1 - dice_score.mean()

In [ ]:
sigmoid = nn.Sigmoid()

# ---- Models, Loss, Optimizer ----
encoder = TerraMindEncoder()
decoder = UNet2D()

criterion = DiceLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)

device = "cuda" if torch.cuda.is_available() else "cpu"
encoder.to(device)
decoder.to(device)



# ---- Training Loop ----
epochs = 10
for epoch in range(epochs):
    encoder.train()
    decoder.train()
    running_loss = 0.0

    for inputs, targets in dataset:
        inputs, targets = inputs.to(device), targets.to(device)

        # Forward pass (explicit encoder + decoder)
        z = encoder(inputs)        # latent features
        logits = decoder(z) # reconstruction

        predictions = sigmoid(logits) 
        predictions = torch.argmax(predictions, dim=1)
        print(predictions.shape, targets.shape)

        loss = criterion(predictions, targets)

        # Backward pass
        optimizer.zero_grad()
        loss.requires_grad = True
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(dataset)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 1/10 - Loss: -0.1187
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 2/10 - Loss: -0.1116
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 3/10 - Loss: -0.1272
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 4/10 - Loss: -0.1311
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 5/10 - Loss: -0.1337
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 6/10 - Loss: -0.1102
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 7/10 - Loss: -0.1358
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 8/10 - Loss: -0.1314
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 9/10 - Loss: -0.1004
torch.Size([1, 224, 224]) torch.Size([1, 224, 224])
Epoch 10/10 - Loss: -0.1162
